# 02 — Graph Construction

This notebook demonstrates how tabular IDS records are converted into
PyTorch Geometric graph objects for the GT-ADF framework.

Two construction modes are shown:
- **k-NN graph**: nodes connected by feature-space similarity
- **EV topology graph**: explicit EV / CS / GC heterogeneous graph (Eqs. 5–7)

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch

from src.data.graph_builder import GraphBuilder
from src.data.preprocessor import Preprocessor
%matplotlib inline

## 1. k-NN Graph from IDS records

Each sliding window of 10 consecutive flow records becomes one graph. Edges connect the k=5 most similar flows in feature space.

In [ ]:
np.random.seed(42)
N, F = 50, 20
X = np.random.randn(N, F)
y = np.random.randint(0, 3, N)

builder = GraphBuilder(k_neighbors=5, batch_size=10, edge_method='knn')
graphs = builder.build_graphs(X, y)

print(f'Built {len(graphs)} graphs')
g = graphs[0]
print(f'Nodes: {g.num_nodes} | Edges: {g.edge_index.shape[1]} | Features: {g.x.shape[-1]}')
print(f'Graph label: {g.y.item()} | Node labels: {g.node_labels.tolist()}')

In [ ]:
# Visualize first graph
G = nx.Graph()
G.add_nodes_from(range(g.num_nodes))
edges = g.edge_index.t().tolist()
G.add_edges_from(edges)

colors = ['#e63946' if l > 0 else '#a8dadc' for l in g.node_labels.tolist()]
fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_spring(G, ax=ax, node_color=colors, node_size=400,
               edge_color='gray', alpha=0.8, with_labels=True)
ax.set_title('k-NN Graph (red=attack, blue=benign)')
plt.tight_layout()
plt.show()

## 2. EV Topology Graph (Eqs. 5–7)

Explicit heterogeneous graph with EV nodes, charging station nodes, and grid component nodes. Edges encode power-flow and data-exchange relations.

In [ ]:
n_ev, n_cs, n_gc = 4, 2, 1
ev_feats = np.random.randn(n_ev, 10)
cs_feats = np.random.randn(n_cs, 10)
gc_feats = np.random.randn(n_gc, 10)

topo_graph = builder.build_ev_topology_graph(
    n_ev, n_cs, n_gc, ev_feats, cs_feats, gc_feats
)
print(f'Topology graph: {topo_graph.num_nodes} nodes, {topo_graph.edge_index.shape[1]} edges')
print(f'Node types: {topo_graph.node_type.tolist()}  (0=EV, 1=CS, 2=GC)')

In [ ]:
G2 = nx.DiGraph()
G2.add_nodes_from(range(topo_graph.num_nodes))
G2.add_edges_from(topo_graph.edge_index.t().tolist())

type_colors = {0: '#e63946', 1: '#457b9d', 2: '#2d6a4f'}
node_colors = [type_colors[t.item()] for t in topo_graph.node_type]
node_labels = {i: f'EV{i}' if t==0 else (f'CS{i-n_ev}' if t==1 else f'GC{i-n_ev-n_cs}')
               for i, t in enumerate(topo_graph.node_type.tolist())}

fig, ax = plt.subplots(figsize=(7, 5))
pos = nx.spring_layout(G2, seed=42)
nx.draw(G2, pos=pos, ax=ax, node_color=node_colors, node_size=600,
        labels=node_labels, font_color='white', font_weight='bold',
        edge_color='gray', arrows=True, arrowsize=15)

from matplotlib.patches import Patch
legend = [Patch(facecolor='#e63946', label='EV'),
          Patch(facecolor='#457b9d', label='Charging Station'),
          Patch(facecolor='#2d6a4f', label='Grid Component')]
ax.legend(handles=legend, loc='upper right')
ax.set_title('EV Charging Network Topology Graph')
plt.tight_layout()
plt.show()